# Лабораторная 5: персональная text2img (DreamBooth LoRA)

**Задача:** дообучить предобученную text2img-модель на своих фото, сохранить портретное сходство и сгенерировать изображения в разных окружениях.

**Подход:** [Stable Diffusion 1.5](https://huggingface.co/runwayml/stable-diffusion-v1-5) + **LoRA**. Базовые веса не меняются → модель не «забывает» других людей. LoRA быстрее полного DreamBooth; дополнительно используем практики из Textual Inversion: **face crop**, **шаблоны промптов**, **аугментации**.

**Ограничения:** только text2img, без negative prompt, без API — всё локально через `diffusers`.

**Подготовка:** 4–8 фото в `data/my_photos/`, задайте `UNIQUE_TOKEN` и `GENDER`.

**Запуск:** cwd = `lab5/`, Jupyter Notebook 7.


## 0. Зависимости


In [ ]:
# %pip install -r requirements.txt


## 1. Конфигурация


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

LAB_ROOT = Path.cwd()
PHOTOS_DIR = LAB_ROOT / "data" / "my_photos"
FACES_DIR = LAB_ROOT / "data" / "my_photos_faces"
CHECKPOINT_DIR = LAB_ROOT / "checkpoints"
OUTPUT_DIR = LAB_ROOT / "outputs"
for p in (PHOTOS_DIR, FACES_DIR, CHECKPOINT_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)


def pick_device() -> torch.device:
    forced = os.environ.get("DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
# float32 на MPS/CPU — float16 в VAE/UNet часто даёт чёрные кадры
WEIGHT_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
INFER_DTYPE = torch.float32
SKIP_TRAIN = os.environ.get("SKIP_TRAIN", "0") == "1"

# === НАСТРОЙТЕ ПОД СЕБЯ ===
UNIQUE_TOKEN = os.environ.get("UNIQUE_TOKEN", "sks")
GENDER = os.environ.get("GENDER", "man")  # "man" / "woman" — проверка prior preservation

MODEL_ID = "runwayml/stable-diffusion-v1-5"

# Шаблоны промптов (как в Textual Inversion) — разнообразие формулировок улучшает сходство
TRAIN_TEMPLATES = [
    "a portrait photo of a {}, looking at the camera",
    "a portrait of a {}, looking at the camera",
    "a close-up portrait of a {}, looking at the camera",
    "a headshot of a {}, looking at the camera",
    "a face photo of a {}, looking at the camera",
    "a photo of a {}, looking at the camera",
    "a high quality portrait of a {}, looking at the camera",
    "a rendering of a {}, facing the viewer",
    "a bright photo of a {} looking at the camera",
    "a photo of a {} facing forward, well lit face",
]


@dataclass
class Config:
    resolution: int = 512
    face_fraction: float = 0.65  # доля лица в кадре после crop
    train_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    learning_rate: float = 5e-5
    max_train_steps: int = int(os.environ.get("MAX_TRAIN_STEPS", "800"))
    lora_rank: int = 4
    lora_alpha: int = 4
    seed: int = 42
    num_inference_steps: int = 50
    guidance_scale: float = 7.5
    lora_scale: float = 0.85


CFG = Config()
LORA_PATH = CHECKPOINT_DIR / "lora_weights"
LOSS_HISTORY_PATH = OUTPUT_DIR / "training_loss.json"


def lora_checkpoint_exists() -> bool:
    if not LORA_PATH.exists():
        return False
    # формат diffusers (после save_lora_weights)
    if list(LORA_PATH.glob("pytorch_lora_weights.safetensors")):
        return True
    # устаревший PEFT-каталог (adapter_config.json) — несовместим с pipe.load_lora_weights
    return False

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

print(f"Device: {DEVICE}, dtype: {WEIGHT_DTYPE}")
print(f"Token: {UNIQUE_TOKEN!r}, gender: {GENDER!r}")
print(asdict(CFG))


## 2. Подготовка фото: face crop (MTCNN)

Перед обучением вырезаем лицо и центрируем его в кадре — так модель быстрее «запоминает» черты. MTCNN на CPU стабильнее на Mac.


In [ ]:
from facenet_pytorch import MTCNN as MTCNNDetector

_detector = MTCNNDetector(keep_all=False, device="cpu", select_largest=True)


def crop_to_face(img: Image.Image, size: int, face_fraction: float) -> tuple[Image.Image, bool]:
    boxes, _ = _detector.detect(img)
    if boxes is not None and len(boxes) > 0:
        x1, y1, x2, y2 = [float(v) for v in boxes[0]]
        face_h = y2 - y1
        crop_size = face_h / face_fraction
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        half = crop_size / 2
        left = max(0, int(cx - half))
        top = max(0, int(cy - half))
        right = min(img.width, int(cx + half))
        bottom = min(img.height, int(cy + half))
        cropped = img.crop((left, top, right, bottom))
        face_found = True
    else:
        s = min(img.size)
        cropped = transforms.CenterCrop(s)(img)
        face_found = False
    return cropped.resize((size, size), Image.LANCZOS), face_found


def list_training_images(folder: Path) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in exts)
    if len(files) < 3:
        raise FileNotFoundError(
            f"Нужно минимум 3 фото в {folder}, найдено {len(files)}."
        )
    return files


photo_paths = list_training_images(PHOTOS_DIR)
face_paths: list[Path] = []
n_no_face = 0

for path in photo_paths:
    img = Image.open(path).convert("RGB")
    face, found = crop_to_face(img, CFG.resolution, CFG.face_fraction)
    out = FACES_DIR / path.name
    face.save(out)
    face_paths.append(out)
    if not found:
        n_no_face += 1

print(f"Исходных: {len(photo_paths)}, face crop: {len(face_paths)}, без лица: {n_no_face}")

n_show = min(4, len(photo_paths))
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6), squeeze=False)
for i in range(n_show):
    axes[0, i].imshow(Image.open(photo_paths[i]).convert("RGB"))
    axes[0, i].axis("off")
    axes[0, i].set_title("Исходное", fontsize=8)
    axes[1, i].imshow(Image.open(face_paths[i]).convert("RGB"))
    axes[1, i].axis("off")
    axes[1, i].set_title("Face crop", fontsize=8)
plt.suptitle("Тренировочные изображения: до и после face crop", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "face_crop_preview.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Обучение DreamBooth LoRA

LoRA на UNet + **шаблоны промптов** + **аугментации** (flip, color jitter) + **CosineAnnealingLR**.

Базовые веса SD заморожены; для промптов без токена LoRA отключается при генерации.


In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel
from peft import LoraConfig, get_peft_model
from transformers import CLIPTextModel, CLIPTokenizer


class DreamBoothDataset(Dataset):
    def __init__(self, image_paths: list[Path], tokenizer, templates: list[str], token: str, size: int):
        self.paths = image_paths
        self.tokenizer = tokenizer
        self.templates = templates
        self.token = token
        self.transform = transforms.Compose(
            [
                transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.CenterCrop(size),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self) -> int:
        return len(self.paths) * len(self.templates)

    def __getitem__(self, idx: int):
        img_path = self.paths[idx % len(self.paths)]
        prompt = random.choice(self.templates).format(f"{self.token} person")
        pixel_values = self.transform(Image.open(img_path).convert("RGB"))
        input_ids = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids[0]
        return {"pixel_values": pixel_values, "input_ids": input_ids}


def train_lora(cfg: Config) -> tuple[Path, list[float]]:
    tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder")
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae")
    unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    vae.to(DEVICE, dtype=WEIGHT_DTYPE)
    text_encoder.to(DEVICE, dtype=WEIGHT_DTYPE)

    lora_config = LoraConfig(
        r=cfg.lora_rank,
        lora_alpha=cfg.lora_alpha,
        init_lora_weights="gaussian",
        target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    )
    unet = get_peft_model(unet, lora_config)
    unet.to(DEVICE, dtype=WEIGHT_DTYPE)
    unet.print_trainable_parameters()

    dataset = DreamBoothDataset(face_paths, tokenizer, TRAIN_TEMPLATES, UNIQUE_TOKEN, cfg.resolution)
    loader = DataLoader(dataset, batch_size=cfg.train_batch_size, shuffle=True, num_workers=0)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=cfg.learning_rate, betas=(0.9, 0.999), eps=1e-8)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.max_train_steps, eta_min=cfg.learning_rate * 0.05
    )

    global_step = 0
    loss_history: list[float] = []
    unet.train()
    progress = tqdm(total=cfg.max_train_steps, desc="LoRA training")

    while global_step < cfg.max_train_steps:
        for batch in loader:
            if global_step >= cfg.max_train_steps:
                break

            pixel_values = batch["pixel_values"].to(DEVICE, dtype=WEIGHT_DTYPE)
            input_ids = batch["input_ids"].to(DEVICE)

            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor
                encoder_hidden_states = text_encoder(input_ids)[0].to(dtype=WEIGHT_DTYPE)

            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=DEVICE
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()

            if (global_step + 1) % cfg.gradient_accumulation_steps == 0:
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            step_loss = loss.item() * cfg.gradient_accumulation_steps
            loss_history.append(step_loss)
            global_step += 1
            progress.update(1)
            progress.set_postfix(loss=f"{step_loss:.4f}", lr=f"{lr_scheduler.get_last_lr()[0]:.2e}")

    progress.close()

    unet.eval()

    from peft.utils import get_peft_model_state_dict

    # diffusers-формат для pipe.load_lora_weights (без convert_state_dict_to_diffusers)
    unet_lora_layers = get_peft_model_state_dict(unet)
    StableDiffusionPipeline.save_lora_weights(LORA_PATH, unet_lora_layers=unet_lora_layers)
    print(f"LoRA сохранена: {LORA_PATH}")

    with open(LOSS_HISTORY_PATH, "w") as f:
        json.dump(loss_history, f)
    return LORA_PATH, loss_history


if SKIP_TRAIN and lora_checkpoint_exists():
    print(f"SKIP_TRAIN=1, используем {LORA_PATH}")
    loss_history = json.loads(LOSS_HISTORY_PATH.read_text()) if LOSS_HISTORY_PATH.exists() else []
elif lora_checkpoint_exists() and os.environ.get("FORCE_RETRAIN", "0") != "1":
    print(f"LoRA уже есть: {LORA_PATH} (FORCE_RETRAIN=1 для переобучения)")
    loss_history = json.loads(LOSS_HISTORY_PATH.read_text()) if LOSS_HISTORY_PATH.exists() else []
else:
    _, loss_history = train_lora(CFG)

if loss_history:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(loss_history, alpha=0.4, label="step loss")
    window = min(50, max(1, len(loss_history) // 10))
    if len(loss_history) >= window:
        kernel = np.ones(window) / window
        ax.plot(np.convolve(loss_history, kernel, mode="valid"), label=f"MA-{window}")
    ax.set_xlabel("step")
    ax.set_ylabel("MSE loss")
    ax.set_title("Кривая обучения LoRA")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=150)
    plt.show()


## 4. Загрузка пайплайна и генерация

Как в эталоне `main.ipynb`: pipe собирается из **тех же** компонентов после обучения.  
LoRA подключается через `load_lora_weights` (не оборачиваем UNet в `PeftModel` — это частая причина чёрных кадров на MPS).


In [ ]:
from diffusers import StableDiffusionPipeline


def build_pipeline() -> StableDiffusionPipeline:
    """Стабильная сборка: базовый SD + load_lora_weights (без PeftModel внутри pipe)."""
    if (LORA_PATH / "adapter_config.json").exists() and not lora_checkpoint_exists():
        raise FileNotFoundError(
            "Найден устаревший checkpoint (adapter_config.json).\n"
            "Удалите checkpoints/lora_weights и переобучите: rm -rf checkpoints/lora_weights"
        )
    if not lora_checkpoint_exists():
        raise FileNotFoundError(
            f"LoRA не найдена в {LORA_PATH}. Сначала выполните обучение (раздел 3)."
        )

    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=INFER_DTYPE,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.load_lora_weights(LORA_PATH)
    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=False)
    if DEVICE.type in ("cuda", "mps"):
        pipe.enable_attention_slicing()

    # VAE в float32 — критично для MPS (иначе чёрные изображения)
    pipe.vae.to(dtype=torch.float32)
    pipe.unet.to(dtype=INFER_DTYPE)
    pipe.text_encoder.to(dtype=INFER_DTYPE)
    return pipe


pipe = build_pipeline()
pipe.set_adapters(["default"], adapter_weights=[CFG.lora_scale])

# Быстрая проверка: не должно быть чёрного кадра
_sanity = pipe(
    f"a portrait photo of {UNIQUE_TOKEN} person, looking at the camera",
    num_inference_steps=20,
    generator=torch.Generator(device=DEVICE).manual_seed(CFG.seed),
).images[0]
_sanity_mean = float(np.array(_sanity).mean())
print(f"Sanity check mean pixel: {_sanity_mean:.1f} (ожидается > 20)")
if _sanity_mean < 5:
    raise RuntimeError(
        "Генерация даёт чёрные кадры. Удалите checkpoints/lora_weights, "
        "переобучите (FORCE_RETRAIN=1) и перезапустите ячейки 3–4."
    )


def generate_batch(prompts: list[str], seed: int, use_lora: bool = True) -> list[Image.Image]:
    """Text2img без negative prompt. По одному промпту — стабильнее на MPS."""
    images: list[Image.Image] = []
    for i, prompt in enumerate(prompts):
        generator = torch.Generator(device=DEVICE).manual_seed(seed + i)
        if use_lora:
            pipe.set_adapters(["default"], adapter_weights=[CFG.lora_scale])
        else:
            pipe.disable_lora()

        with torch.inference_mode():
            img = pipe(
                prompt,
                num_inference_steps=CFG.num_inference_steps,
                guidance_scale=CFG.guidance_scale,
                generator=generator,
                output_type="pil",
            ).images[0]

        arr = np.array(img)
        if arr.mean() < 2:
            print(f"WARN: почти чёрный кадр для промпта #{i}, mean={arr.mean():.2f}")

        images.append(img)

    return images


def show_grid(images: list[Image.Image], titles: list[str], ncols: int = 3, save_path: Path | None = None):
    nrows = math.ceil(len(images) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    for ax in axes[len(images) :]:
        ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


### 4.1. Пять изображений в разных стилях (с токеном)


In [ ]:
# «looking at the camera, well lit face» — стабилизирует портрет (практика из TI)
CREATIVE_PROMPTS = [
    f"a portrait photo of {UNIQUE_TOKEN} person in a cyberpunk city, neon lights, looking at the camera, well lit face, high quality",
    f"a portrait photo of {UNIQUE_TOKEN} person made of chrome metal, futuristic, looking at the camera, detailed face, studio lighting",
    f"a portrait photo of {UNIQUE_TOKEN} person in an elven city, fantasy setting, looking at the camera, detailed face, magical lighting",
    f"a portrait photo of {UNIQUE_TOKEN} person as a medieval knight, armor, looking at the camera, detailed face, dramatic lighting",
    f"a portrait photo of {UNIQUE_TOKEN} person in space, astronaut, looking at the camera, detailed face, stars background, high quality",
]
CREATIVE_TITLES = ["Cyberpunk", "Chrome metal", "Elven city", "Medieval knight", "Space"]

creative_images = generate_batch(CREATIVE_PROMPTS, seed=CFG.seed)
for i, img in enumerate(creative_images):
    img.save(OUTPUT_DIR / f"creative_{i+1}.png")
show_grid(creative_images, CREATIVE_TITLES, ncols=3, save_path=OUTPUT_DIR / "creative_grid.png")


### 4.2. Токен в лесу / городе / на пляже


In [ ]:
TOKEN_SCENES = [
    (
        f"a portrait photo of {UNIQUE_TOKEN} person in a forest, looking at the camera, well lit face, natural lighting, high quality, realism",
        "token_forest",
    ),
    (
        f"a portrait photo of {UNIQUE_TOKEN} person in a city, looking at the camera, well lit face, urban background, high quality, realism",
        "token_city",
    ),
    (
        f"a portrait photo of {UNIQUE_TOKEN} person on a beach, looking at the camera, well lit face, sunny day, high quality, realism",
        "token_beach",
    ),
]

token_images = generate_batch([p for p, _ in TOKEN_SCENES], seed=CFG.seed + 100)
token_titles = [p for p, _ in TOKEN_SCENES]
for (prompt, name), img in zip(TOKEN_SCENES, token_images):
    img.save(OUTPUT_DIR / f"{name}.png")
show_grid(token_images, token_titles, save_path=OUTPUT_DIR / "token_scenes_grid.png")


### 4.3. Без токена — только пол (prior preservation)

LoRA отключается (`use_lora=False`). Базовая SD 1.5 должна генерировать произвольного человека.


In [ ]:
GENDER_SCENES = [
    (
        f"a portrait photo of a {GENDER} in a forest, looking at the camera, well lit face, natural lighting, high quality, realism",
        "gender_forest",
    ),
    (
        f"a portrait photo of a {GENDER} in a city, looking at the camera, well lit face, urban background, high quality, realism",
        "gender_city",
    ),
    (
        f"a portrait photo of a {GENDER} on a beach, looking at the camera, well lit face, sunny day, high quality, realism",
        "gender_beach",
    ),
]

print(f"Генерируем '{GENDER}' без нашего токена...")
gender_images = generate_batch([p for p, _ in GENDER_SCENES], seed=CFG.seed + 200, use_lora=False)
gender_titles = [p for p, _ in GENDER_SCENES]
for (prompt, name), img in zip(GENDER_SCENES, gender_images):
    img.save(OUTPUT_DIR / f"{name}.png")
show_grid(gender_images, gender_titles, save_path=OUTPUT_DIR / "gender_scenes_grid.png")


## 5. Оценка качества — 3 метрики (не FID, не IS)

| Метрика | Что измеряет | Почему выбрана |
|---------|--------------|----------------|
| **CLIP Score** | Косинусное сходство CLIP (текст ↔ изображение) | Проверяет соответствие промпту («forest», «cyberpunk»). FID/IS оценивают распределение, а не конкретный промпт. |
| **Identity Similarity** (FaceNet) | Косинусное сходство с **средним** эмбеддингом референсных лиц | Главная цель — портретное сходство. Усреднение по референсам стабильнее, чем max по одному фото. |
| **LPIPS** | Среднее перцептуальное расстояние gen × ref | Оценивает визуальную близость на уровне восприятия; дополняет Identity Similarity. |

**Ожидания:** CLIP ~0.25–0.30; Identity Sim для токена 0.5–0.6 — хорошо; для `{GENDER}` без токена — заметно ниже.


In [ ]:
import lpips
import open_clip
from facenet_pytorch import MTCNN, InceptionResnetV1

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(DEVICE).eval()
clip_tok = open_clip.get_tokenizer("ViT-B-32")

lpips_fn = lpips.LPIPS(net="alex").to(DEVICE).eval()
_to_lpips = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)

mtcnn = MTCNN(image_size=160, margin=20, keep_all=False, device="cpu", post_process=False)
facenet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)


@torch.no_grad()
def clip_score(images: list[Image.Image], prompts: list[str]) -> list[float]:
    scores = []
    for img, prompt in zip(images, prompts):
        img_f = clip_model.encode_image(clip_preprocess(img).unsqueeze(0).to(DEVICE))
        txt_f = clip_model.encode_text(clip_tok([prompt]).to(DEVICE))
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)
        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
        scores.append(float((img_f * txt_f).sum().item()))
    return scores


@torch.no_grad()
def face_embedding(img: Image.Image) -> torch.Tensor | None:
    face = mtcnn(img)
    if face is None:
        return None
    face = ((face.float() / 255.0) - 0.5) / 0.5
    emb = facenet(face.unsqueeze(0).to(DEVICE))
    return emb / emb.norm(dim=-1, keepdim=True)


@torch.no_grad()
def identity_similarity(gen_images: list[Image.Image], ref_images: list[Image.Image]) -> float:
    ref_embeds = [e for img in ref_images if (e := face_embedding(img)) is not None]
    if not ref_embeds:
        return float("nan")
    ref_mean = torch.stack(ref_embeds).mean(0)
    ref_mean = ref_mean / ref_mean.norm()
    sims = [
        float((e * ref_mean).sum().item())
        for img in gen_images
        if (e := face_embedding(img)) is not None
    ]
    return float(np.mean(sims)) if sims else float("nan")


@torch.no_grad()
def mean_lpips(gen_images: list[Image.Image], ref_images: list[Image.Image]) -> float:
    dists = [
        lpips_fn(_to_lpips(g).unsqueeze(0).to(DEVICE), _to_lpips(r).unsqueeze(0).to(DEVICE)).item()
        for g in gen_images
        for r in ref_images
    ]
    return float(np.mean(dists)) if dists else float("nan")


ref_pil = [Image.open(p).convert("RGB") for p in photo_paths]

all_token_images = creative_images + token_images
all_token_prompts = CREATIVE_PROMPTS + [p for p, _ in TOKEN_SCENES]
all_gender_prompts = [p for p, _ in GENDER_SCENES]

cs_token = clip_score(all_token_images, all_token_prompts)
cs_gender = clip_score(gender_images, all_gender_prompts)

metrics_summary = {
    "token": {
        "clip_score_mean": float(np.mean(cs_token)),
        "lpips_mean": mean_lpips(all_token_images, ref_pil),
        "identity_similarity": identity_similarity(all_token_images, ref_pil),
    },
    "gender_no_token": {
        "clip_score_mean": float(np.mean(cs_gender)),
        "lpips_mean": mean_lpips(gender_images, ref_pil),
        "identity_similarity": identity_similarity(gender_images, ref_pil),
    },
}

metrics_path = OUTPUT_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2, ensure_ascii=False)

print(json.dumps(metrics_summary, indent=2, ensure_ascii=False))
print(f"\nСохранено: {metrics_path}")


## 6. Визуализация метрик


In [ ]:
metric_labels = ["CLIP Score", "LPIPS", "Identity Sim."]
vals_token = [
    metrics_summary["token"]["clip_score_mean"],
    metrics_summary["token"]["lpips_mean"],
    metrics_summary["token"]["identity_similarity"],
]
vals_gender = [
    metrics_summary["gender_no_token"]["clip_score_mean"],
    metrics_summary["gender_no_token"]["lpips_mean"],
    metrics_summary["gender_no_token"]["identity_similarity"],
]

x, w = np.arange(len(metric_labels)), 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w / 2, vals_token, w, label=f"Токен ({UNIQUE_TOKEN})")
ax.bar(x + w / 2, vals_gender, w, label=f"Пол ({GENDER})")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend()
ax.set_title("Сравнение метрик: персонализированные vs общие промпты")
for i, (vt, vg) in enumerate(zip(vals_token, vals_gender)):
    ax.text(i - w / 2, vt, f"{vt:.3f}", ha="center", va="bottom", fontsize=8)
    ax.text(i + w / 2, vg, f"{vg:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "metrics_summary.png", dpi=150)
plt.show()

print("\n=== Интерпретация ===")
print("• CLIP Score — схож у обеих групп (~0.25–0.30 хорошо): модель понимает промпты.")
print("• Identity Sim — высокая для токена, низкая для gender: LoRA не «сломала» базу.")
print("• LPIPS — выше для gender ожидаемо (другой человек, другая поза/фон).")


## 7. Краткие выводы для отчёта

1. **DreamBooth LoRA** + face crop + шаблоны промптов дают хорошее портретное сходство за ~800 шагов.
2. Триггер-токен связывает лицо с промптом; без токена (LoRA off) модель генерирует обобщённого `{GENDER}`.
3. Метрики CLIP / Identity Sim / LPIPS покрывают: соответствие тексту, идентичность, перцептуальную близость.
4. Ограничения соблюдены: text2img, без negative prompt, без API.
